# FTP Curve Model — Single Run & Scenario Testing

Builds the base yield curve and liquidity-premium curve from t-bill,
bond, and deposit market data, combines them into a full FTP term curve,
exports the reporting workbooks the Excel reference layer consumes, and
(section 5) runs the same build across a set of parallel rate shocks to
sanity-check curve sensitivity.

Flow: `Market Data -> Base Curve (Nelson-Siegel fit) -> 3m Shift +
Scenario Shock -> Liquidity Premium (Nelson-Siegel fit) -> Full FTP
Curve -> Export`

All calculation logic lives in **`ftp_curve_model.py`** (imported below)
— keep that file in the same folder as this notebook. This notebook only
does the "run the model today" work: point it at input files, choose
config values, run it, export it, and compare scenarios. **To change how
the model itself works, edit `ftp_curve_model.py`, not this notebook.**

In [ ]:
import logging
import sys
sys.path.insert(0, r".\modules")
from ftp_curve_model import (
    CurveConfig,
    DataObject,
    CurveModelObject,
    ExportObject,
    run_scenario_suite,
    compare_scenarios,
    ensure_export_dir,
    list_deposit_rate_columns,
)

## 1. Run Identity & Logging

Three constants that get reused across every section below, defined
once here rather than wherever they first happen to be needed:

- **`EXPORT_DIR`** — where the run log (`ftp_model.log`) is written, and
  the **default** export location for every output below (each output
  can still be pointed somewhere else individually — see sections 2 and
  4).
- **`MODEL_VERSION`** — tags every exported file and feeds into
  `data.prepare_data()` and `CurveConfig` below, so every artifact from
  this run is traceable to the same version string.
- **`reporting_month`** — the one thing you type by hand each month;
  used to build this run's export folder paths in sections 2 and 3.

In [ ]:
EXPORT_DIR = "export"
MODEL_VERSION = "v2-2"
reporting_month = "202604"   # yyyymm

ensure_export_dir(EXPORT_DIR)

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format=(
        "%(asctime)s | "
        "%(levelname)-8s | "
        "%(module)s | "
        "%(funcName)s | "
        "%(message)s"
    ),
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(
            f"{EXPORT_DIR}/ftp_model.log",
            mode="a"
        )
    ]
)

## 2. Load & Prepare Market Data

### Deposit rate source

`DEPOSIT_PRIMARY_SOURCE` must exactly match a column name in
`DEPOSITS_FILE` — the cell below prints what's actually there before you
pick. Up to 2 `DEPOSIT_FALLBACK_SOURCES` are tried, in list order, for
any row missing the primary (e.g. `["FNB", "Standard Bank"]` tries FNB
first, then Standard Bank, only where Nedbank itself is missing).

### Market data export location

One switch and one base folder for all of t-bills, bonds, and deposits
together: `EXPORT_MARKET_DATA` (off by default) and
`MARKET_DATA_EXPORT_DIR`. Each source still gets its own labelled
subfolder underneath it (`tbills/`, `bonds/`, `deposits/`) — that's a
`DataObject` requirement (each source takes its own export path), not a
choice this notebook is making, so we derive all three from the one
base folder rather than typing three unrelated paths by hand. Covers
both the raw per-window market data (`tbills_latest.xlsx`, `*_ew_1m`,
`*_ew_6m`, `bonds_latest.xlsx`, `deposits.xlsx`, ...) and the
prepared/bucketed data (`tbills_df.xlsx`, `tbill_bkt_*.xlsx`,
`bond_bkt_*.xlsx`, `deposit_bkt_*.xlsx`).

In [ ]:
# --- Data Layer ---
DEPOSITS_FILE = "input/deposit_rates_compare_ii.csv"
print("Available deposit rate columns:", list_deposit_rate_columns(DEPOSITS_FILE))

DEPOSIT_PRIMARY_SOURCE = "fnb"
DEPOSIT_FALLBACK_SOURCES = ["standard"]

In [ ]:
EXPORT_MARKET_DATA = False
MARKET_DATA_EXPORT_DIR = fr".\{EXPORT_DIR}\{reporting_month}\market_data"

# One subfolder per source, all under the single base folder above -
# these three names are what DataObject actually expects (see section
# below), not a daily/weekly/monthly split.
tbills_output_folder = fr"{MARKET_DATA_EXPORT_DIR}\tbills"
bonds_output_folder = fr"{MARKET_DATA_EXPORT_DIR}\bonds"
deposits_output_folder = fr"{MARKET_DATA_EXPORT_DIR}\deposits"

In [ ]:
data = DataObject(
    tbills_file="input/treasury_bills_data_ii.csv",
    bonds_file="input/bond_yields_data_ii.csv",
    deposits_file=DEPOSITS_FILE,
    deposit_primary_source=DEPOSIT_PRIMARY_SOURCE,
    deposit_fallback_sources=DEPOSIT_FALLBACK_SOURCES,
    export_tbills=EXPORT_MARKET_DATA, tbills_export_dir=tbills_output_folder,
    export_bonds=EXPORT_MARKET_DATA, bonds_export_dir=bonds_output_folder,
    export_deposits=EXPORT_MARKET_DATA, deposits_export_dir=deposits_output_folder,
)
data.load_data()
data.set_cutoff_dates(
    tbills="2026-04-27",
    bonds="2026-04-08",
    deposits="2025-12-31",
    snapshot_window="Latest",
)
data.prepare_data(model_version=MODEL_VERSION)
data.print_specs()

## 3. Base Case Curve

### Curve settings for this run

- **`apply_3m_shift`** — the deposit-vs-3-month-T-bill "observability"
  spread is added to the base curve by default. Set `False` to build the
  curve *without* it (only the scenario `parallel_shift_bps` shock still
  applies) — the spread itself is still computed and logged either way,
  just not added when this is off.
- **`export_dir`** (`FTP_OUTPUT_EXPORT_DIR` below) — where this curve's
  own FTP outputs go (`ftp_base_df`, the bucketed curve, curve output,
  summary — section 4); independent from `MARKET_DATA_EXPORT_DIR` above.
- **`visualize`** — set `False` to skip all plots and diagnostic
  printouts for a fast, quiet run (matters most in section 5, where the
  same build runs three times over).

In [ ]:
ftp_output_folder = fr".\{EXPORT_DIR}\{reporting_month}\ftp"

APPLY_3M_SHIFT = True
FTP_OUTPUT_EXPORT_DIR = ftp_output_folder

### Curve fit parameters & LP stress overlay

Nelson-Siegel penalty weights for the base curve (`BASE_LAMBDA_*`) and
the liquidity-premium curve (`LP_LAMBDA_*`), plus the liquidity premium's
long-end cap (`MAX_LP_DEC`, a decimal rate — `0.003` = 30bps). Defaults
below match `CurveConfig`'s own; change them here rather than passing
different values every call.

`APPLY_LP_STRESS` (on by default) adds a tenor-dependent stress overlay
on top of the structural (deposit-minus-base) liquidity premium before
the NS curve is fit to it — `LP_STRESS_ALPHA` is the maximum proportional
uplift (`0.30` = up to +30%), `LP_STRESS_TAU_S` the build-up horizon in
years, `LP_MAX_STRESS_BPS` a hard cap on the stress add-on itself (not
the total LP).

**If you set `APPLY_LP_STRESS = False`,** also check `lp_fit_target`
(currently left at `CurveConfig`'s default, `"Stress_LP"`) — with stress
off, that column is all zeros and the LP curve fits to flat zero. Pass
`lp_fit_target="Total_LP"` (or `"Total_LP_Monotonic"`) into `CurveConfig`
below to fit the structural premium on its own instead.

In [ ]:
# Base curve (Nelson-Siegel) fit penalties
BASE_LAMBDA_LEVEL = 0.001
BASE_LAMBDA_CURVATURE = 0.002
BASE_LAMBDA_MONO = 2.0

# Liquidity premium curve (Nelson-Siegel) fit penalties + long-end cap
MAX_LP_DEC = 0.003
LP_LAMBDA_LEVEL = 0.01
LP_LAMBDA_CURVATURE = 0.002
LP_LAMBDA_MONO = 20.0

# Observed LP stress overlay - see markdown above, especially the
# lp_fit_target note if you turn APPLY_LP_STRESS off.
APPLY_LP_STRESS = True
LP_STRESS_ALPHA = 1.5
LP_STRESS_TAU_S = 25.0
LP_MAX_STRESS_BPS = 200.0

In [ ]:
base_config = CurveConfig(
    model_version=MODEL_VERSION,
    run_purpose="Total Business Trial 001",
    data_cutoff_date="2026-04-30",
    export_dir=FTP_OUTPUT_EXPORT_DIR,
    apply_3m_shift=APPLY_3M_SHIFT,
    base_lambda_level=BASE_LAMBDA_LEVEL,
    base_lambda_curvature=BASE_LAMBDA_CURVATURE,
    base_lambda_mono=BASE_LAMBDA_MONO,
    max_lp_dec=MAX_LP_DEC,
    lp_lambda_level=LP_LAMBDA_LEVEL,
    lp_lambda_curvature=LP_LAMBDA_CURVATURE,
    lp_lambda_mono=LP_LAMBDA_MONO,
    apply_lp_stress=APPLY_LP_STRESS,
    lp_stress_alpha=LP_STRESS_ALPHA,
    lp_stress_tau_s=LP_STRESS_TAU_S,
    lp_max_stress_bps=LP_MAX_STRESS_BPS,
)

base_model = CurveModelObject(data, base_config)
base_model.run()

## 4. Export FTP Output

Three independent, optional artifacts, all under `base_config.export_dir`:

| Flag | Output |
|---|---|
| `bucketed` | `Full_FTP_Bucketed_Curve_*.xlsx` — curve resampled onto standard reporting tenors |
| `csv` | `ftp_curve_output_*.xlsx` — full-resolution base/liquidity/FTP rates |
| `summary` | `ftp_model_output_ver_*.xlsx` — governance summary (NS parameters, fit diagnostics, data cutoffs, whether the 3m shift was applied) |

Set any of the three to `False` below to skip it for this run.

In [ ]:
ExportObject(base_model).export_all(bucketed=True, csv=True, summary=True)

## 5. Scenario Testing

Runs the exact same curve build again, once per parallel shock in
`SHOCKS_BPS`, against the same market data (`data`) and the same
`base_config` — only `parallel_shift_bps` changes between runs. Useful
as a fast sanity check: does the curve, and everything downstream of it,
move the way a uniform rate shock should move it?

`run_scenario_suite` reuses `base_config` for every scenario except the
shock itself, and exports each one to its own file set under
`base_config.export_dir` (so set `visualize=False` on `base_config`
above first if you don't want three full sets of diagnostic plots).
`compare_scenarios` then pivots the results into one Tenor × Scenario
table of FTP rates, with a bps-delta-vs-base column per shock — the
fastest way to see whether, say, a +100bps shock actually moved the
10-year point by roughly 100bps.

In [ ]:
SHOCKS_BPS = (0, 100, -100)   # 0 = base case, always include it so compare_scenarios has something to diff against

scenario_results = run_scenario_suite(data, base_config, shocks_bps=SHOCKS_BPS, export=True)
scenario_comparison = compare_scenarios(scenario_results)
scenario_comparison

---

**Also available, not covered in this walkthrough:** `run_lp_parameter_sweep`
in `ftp_curve_model.py` fits the liquidity-premium curve once per
combination in a grid of LP-stage parameters (stress alpha, tau, fit
target, etc.), reusing a single shared base-curve fit across all of
them — useful for calibration work, but a bigger job than a single
notebook run. See its docstring for the full parameter list and the
`LP_SWEEPABLE_PARAMS` restriction it's built around.